# train_classifier_topic — the topicality-VIEW cross-encoder (clf_topic)

Same pairs + recipe as `clf_R`, but the **`topic_first`** representation (conditions/title/summary lead,
eligibility trails) — a deliberately different *view* so it's a **diverse** reranker feature alongside the
eligibility-view `clf_R`, not a redundant clone. Self-consistent (trained AND scored on `topic_first`).


## Setup (Colab — GPU/A100)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q 'transformers[torch]' datasets accelerate scikit-learn pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ── to push clf_topic after training: paste a WRITE token, then the final cell pushes. Delete before sharing. ──
import os
os.environ['HF_TOKEN'] = ''   # <-- paste WRITE token (https://huggingface.co/settings/tokens); leave '' to skip push


In [ ]:
import json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, torch, torch.nn.functional as F
from ctmatch.experiments import ExperimentConfig, load_corpus, build_clf_dataset, log_result
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = ExperimentConfig(data_root=DATA_ROOT, repr_strategy='topic_first')   # TOPICALITY view
print('repr:', cfg.repr_tag(), '| device:', device)


In [ ]:
BASE_MODEL = 'michiyasunaga/BioLinkBERT-large'
OUT_DIR    = cfg.path('models/clf_topic')
HUB_REPO   = cfg.clf_topic_ckpt   # 'semaj83/ctmatch-clf-topic'
LR, EPOCHS, BATCH, FOCAL_GAMMA, WARMUP = 2e-5, 3, 8, 2.0, 0.1
ID2LABEL = {0: 'not_relevant', 1: 'partially_relevant', 2: 'relevant'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}


In [ ]:
def read_pairs(p): return [json.loads(l) for l in open(p)]
train_pairs = read_pairs(cfg.path('data/clf_pairs_train.jsonl'))
val_pairs   = read_pairs(cfg.path('data/clf_pairs_val.jsonl'))
corpus_ids, corpus_fields = load_corpus(cfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
print(f'train {len(train_pairs):,} | val {len(val_pairs):,}')


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
train_ds = build_clf_dataset(train_pairs, id2fields, tokenizer, cfg)   # cfg = topic_first
val_ds   = build_clf_dataset(val_pairs,   id2fields, tokenizer, cfg)
counts = np.bincount([p['label'] for p in train_pairs], minlength=3).astype(float)
w = 1.0 - counts / counts.sum(); class_weights = torch.tensor(w / w.sum(), dtype=torch.float32).to(device)
print('label counts:', counts.astype(int))


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import f1_score, classification_report
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=3, ignore_mismatched_sizes=True).to(device)
model.config.id2label = ID2LABEL; model.config.label2id = LABEL2ID
class FocalLossTrainer(Trainer):
    def __init__(self, class_weights, gamma=2.0, *a, **k):
        super().__init__(*a, **k); self.class_weights = class_weights; self.gamma = gamma
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels'); out = model(**inputs); logits = out.get('logits')
        ce = F.cross_entropy(logits, labels, weight=self.class_weights, reduction='none')
        loss = ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()
        return (loss, out) if return_outputs else loss
def compute_metrics(p):
    pr = p.predictions.argmax(-1)
    return {'macro_f1': f1_score(p.label_ids, pr, average='macro')}


In [ ]:
args = TrainingArguments(output_dir=OUT_DIR, learning_rate=LR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH, warmup_ratio=WARMUP,
    weight_decay=0.01, fp16=True, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
    logging_steps=50, report_to='none', seed=cfg.seed)
trainer = FocalLossTrainer(class_weights=class_weights, gamma=FOCAL_GAMMA, model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer))
trainer.train()


In [ ]:
pred = trainer.predict(val_ds)
print(classification_report(pred.label_ids, pred.predictions.argmax(-1), target_names=list(ID2LABEL.values())))
m = {'macro_f1': float(f1_score(pred.label_ids, pred.predictions.argmax(-1), average='macro'))}
log_result(cfg, experiment='train_classifier_topic', split='val', metrics=m, extra={'base_model': BASE_MODEL})
trainer.save_model(OUT_DIR); tokenizer.save_pretrained(OUT_DIR)
if os.environ.get('HF_TOKEN'):
    model.push_to_hub(HUB_REPO); tokenizer.push_to_hub(HUB_REPO); print('pushed ->', HUB_REPO)
print('saved clf_topic ->', OUT_DIR, '| val macro-F1:', round(m['macro_f1'], 4))
